# Geocoding Experiments: Google Maps API Evaluation

This notebook evaluates the **geolocation accuracy** of T5 model location extraction predictions by:
1. Loading predictions saved from the location extraction model training
2. Converting model predictions to lat/long coordinates using Google Maps API
3. Comparing against manual baseline (human-looked-up coordinates)
4. Computing distance-based metrics to assess practical utility

## 1. Setup and Configuration

### 1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install required packages
!pip install -qU pip setuptools wheel
!pip install --upgrade transformers datasets evaluate rouge-score rapidfuzz

# 4) Make result directories
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction").mkdir(parents=True, exist_ok=True)

# 5) Import utilities from cloned repo (using importlib to handle hyphens in path)
try:
    import importlib.util
    
    spec = importlib.util.spec_from_file_location(
        "file_io",
        "/content/code-satp/models/classification-models/utils/file_io.py"
    )
    file_io = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(file_io)
    
    get_task_results_dir = file_io.get_task_results_dir
    save_dataframe_csv = file_io.save_dataframe_csv
    
    # Define task name for consistent file organization
    TASK_NAME = "location-extraction"
    
    print("="*80)
    print("✅ SETUP COMPLETE")
    print("="*80)
    print(f"📂 Cloned repo: /content/code-satp")
    print(f"📂 Results dir: {get_task_results_dir(TASK_NAME)}")
    print(f"✅ File I/O utilities loaded successfully")
    print("="*80)
    
except Exception as e:
    print("="*80)
    print("⚠️  WARNING: Could not load file_io utilities")
    print(f"Error: {e}")
    print("="*80)
    print("Falling back to local mode - files will be saved to current directory")
    
    # Define fallback functions
    TASK_NAME = "location-extraction"
    
    def get_task_results_dir(task_name):
        return pathlib.Path(f"./results/{task_name}")
    
    def save_dataframe_csv(df, filename, task_name=None):
        output_path = f"./results/{task_name}/{filename}" if task_name else f"./{filename}"
        pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        return output_path
    
    print("="*80)

### 1.2 Local Setup

### 1.3 Import Libraries

In [ ]:
# Core libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

# API and HTTP
import requests

# Visualization
import matplotlib.pyplot as plt

# Colab-specific (will fail gracefully on local)
try:
    from google.colab import userdata
except ImportError:
    userdata = None
    print('⚠️  Not running on Colab - API key should be set manually')


## 2. Helper Functions

### 2.1 Parse Structured Location Output

In [ ]:
def parse_structured_location(text):
    """Parse structured location string into a dictionary."""
    location_dict = {
        'state': None,
        'district': None,
        'village': None,
        'other_locations': None
    }
    
    if not text or text.strip() == '':
        return location_dict
    
    # Split by comma and parse each part
    parts = [part.strip() for part in text.split(',')]
    for part in parts:
        if ':' in part:
            label, value = part.split(':', 1)
            label = label.strip().lower()
            value = value.strip()
            if label in location_dict:
                location_dict[label] = value
    
    return location_dict

# Test the parser with a sample
sample_location = "state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"
parsed = parse_structured_location(sample_location)
print("Sample parsing test:")
print(f"Input: {sample_location}")
print(f"Parsed: {parsed}")

## 3. Geocoding Evaluation: Google Maps API

This section evaluates the **geolocation accuracy** of our T5 model's predictions by:
1. Loading the predictions saved from the location extraction model
2. Converting model predictions to lat/long coordinates using Google Maps API
3. Comparing against the manual baseline (human-looked-up coordinates)
4. Computing distance-based metrics to assess practical utility

### 3.1 Setup: Load Libraries and API Key

In [ ]:
import requests
import pandas as pd

# Google Maps API key
# For Colab: stored securely in Colab Secrets
#   Add your key: Click 🔑 icon → Add secret → Name: 'googlemapsAPI' → Paste key → Enable notebook access
# For local: set as environment variable or replace with your key directly
try:
    from google.colab import userdata
    API_KEY = userdata.get('googlemapsAPI')
except ImportError:
    # Local execution - get from environment variable or set directly
    import os
    API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', None)
    if not API_KEY:
        print("⚠️  No API key found. Set GOOGLE_MAPS_API_KEY environment variable or add API_KEY here")
        # Uncomment and paste your API key for local execution:
        # API_KEY = "YOUR_API_KEY_HERE"

GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

print("✅ Google Maps API configured")
print(f"   API key loaded: {API_KEY[:10]}..." if API_KEY else "⚠️  No API key found - configure API key above")

Using device: cuda
Using device: cuda


### 3.2 Load Saved Predictions

In [ ]:
# Load the predictions saved in Section 5.5
try:
    results_dir = get_task_results_dir(TASK_NAME)
    predictions_path = f"{results_dir}/location_extraction_test_predictions.csv"
    test_predictions = pd.read_csv(predictions_path)
    print(f"✅ Loaded test predictions from: {predictions_path}")
except (NameError, FileNotFoundError):
    # Fallback: try loading from current directory
    test_predictions = pd.read_csv("./location_extraction_test_predictions.csv")
    print("✅ Loaded test predictions from current directory")

print(f"\nDataset size: {len(test_predictions)} incidents")
print(f"Columns: {', '.join(test_predictions.columns)}")
print(f"\nSample predictions:")
display(test_predictions[['prediction', 'ground_truth']].head(3))

### 3.3 Define Geocoding Functions

#### 3.3.1 Get Location Details

In [ ]:
def get_location_details(address, components=None):
    """
    Get location details using Google Geocoding API with optional component filtering.
    
    Args:
        address: Address string (village + other_locations)
        components: Optional dict with 'state' and/or 'district' for filtering
        
    Returns:
        Dict with state, district, subdistrict, town_village, latitude, longitude
    """
    # Build component filter string
    component_parts = ['country:IN']  # Always filter by India
    
    if components:
        if components.get('state'):
            component_parts.append(f"administrative_area_level_1:{components['state']}")
        if components.get('district'):
            component_parts.append(f"administrative_area_level_2:{components['district']}")
    
    params = {
        'address': address,
        'key': API_KEY,
        'components': '|'.join(component_parts)
    }
    
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        # Don't print error for ZERO_RESULTS (expected for some locations)
        if data['status'] != 'ZERO_RESULTS':
            print(f"Geocoding API error: {data['status']}")
        return None

    # Initialize components
    state = district = subdistrict = town_village = None
    latitude = longitude = None
    found_level = None

    # Iterate over results to find the most specific level
    for result in data.get('results', []):
        temp_state = temp_district = temp_subdistrict = temp_town_village = None
        address_components = result['address_components']

        # Map address components
        for component in address_components:
            types = component['types']
            if 'administrative_area_level_1' in types:
                temp_state = component['long_name']
            elif 'administrative_area_level_2' in types:
                temp_district = component['long_name']
            elif 'administrative_area_level_3' in types:
                temp_subdistrict = component['long_name']
            elif 'locality' in types:
                temp_town_village = component['long_name']
            elif 'sublocality' in types and not temp_town_village:
                temp_town_village = component['long_name']

        # Determine the most specific level in this result
        if temp_town_village and found_level not in ['town_village']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = temp_town_village
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'town_village'
        elif temp_subdistrict and found_level not in ['town_village', 'subdistrict']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'subdistrict'
        elif temp_district and found_level not in ['town_village', 'subdistrict', 'district']:
            state = temp_state
            district = temp_district
            subdistrict = None
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'district'

        # Break the loop if the most specific level is found
        if found_level == 'town_village':
            break

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village,
        'latitude': latitude,
        'longitude': longitude
    }

print("✅ Geocoding function defined")

#### 3.3.2 Parse Structured to Address and Components

In [ ]:
def parse_structured_to_address_and_components(structured_location):
    """
    Parse T5 structured output and build address + components for Google API.
    
    Strategy with fallbacks:
    1. Prefer village + other_locations as address (most specific)
    2. Fall back to district if no village/other_locations (district centroid)
    3. Fall back to state if only state available (state centroid)
    4. Use appropriate component filters based on what's in the address
    
    Examples:
    Input: "state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"
    Output: {'address': 'Nilamadgu, Dornapal', 'components': {'state': 'Chhattisgarh', 'district': 'Sukma'}}
    
    Input: "state: Chhattisgarh, district: Sukma" (no village)
    Output: {'address': 'Sukma', 'components': {'state': 'Chhattisgarh'}}
    
    Input: "state: Chhattisgarh" (only state)
    Output: {'address': 'Chhattisgarh', 'components': None}
    """
    if not structured_location or structured_location.strip() == '':
        return None
    
    parsed = parse_structured_location(structured_location)
    
    # Build address from most specific to least specific
    address_parts = []
    
    # Level 1: Village + other_locations (most specific)
    if parsed['village']:
        address_parts.append(parsed['village'])
    if parsed['other_locations']:
        address_parts.append(parsed['other_locations'])
    
    # Level 2: Fallback to district if no village/other_locations
    if not address_parts and parsed['district']:
        address_parts.append(parsed['district'])
    
    # Level 3: Fallback to state if nothing else available
    if not address_parts and parsed['state']:
        address_parts.append(parsed['state'])
    
    # Build component filters (don't filter by what's already in the address)
    components = {}
    
    # If we're using village/other_locations as address, use state+district as filters
    if parsed['village'] or parsed['other_locations']:
        if parsed['state']:
            components['state'] = parsed['state']
        if parsed['district']:
            components['district'] = parsed['district']
    
    # If we're using district as address, only use state as filter
    elif parsed['district'] and address_parts and parsed['district'] in address_parts[0]:
        if parsed['state']:
            components['state'] = parsed['state']
    
    # If we're using state as address, no additional filters needed
    # (Google will understand "Chhattisgarh, India" without extra components)
    
    # If we have nothing to geocode, return None
    if not address_parts:
        return None
    
    return {
        'address': ', '.join(address_parts),
        'components': components if components else None
    }

print("✅ Parser function defined (with district/state fallback)")

### 3.4 Geocode Model Predictions

In [ ]:
# Geocode the T5 model predictions
print(f"Geocoding {len(test_predictions)} model predictions...")
print("This will make 2 API calls per row (model prediction + human baseline)")
print(f"Expected API calls: ~{len(test_predictions) * 2}")
print("="*80)

geocoding_results = []

for idx, row in test_predictions.iterrows():
    # Parse the model's prediction
    parsed = parse_structured_to_address_and_components(row['prediction'])
    
    result = {
        'index': idx,
        'input_text': row['input_text'],
        'model_prediction': row['prediction'],
        'ground_truth': row['ground_truth'],
        'incident_number': row['incident_number'],
        'incident_summary': row['incident_summary'],
    }
    
    # Geocode model prediction
    if parsed:
        location_details = get_location_details(parsed['address'], parsed['components'])
        if location_details:
            result['model_lat'] = location_details['latitude']
            result['model_lon'] = location_details['longitude']
            result['model_geocoded_state'] = location_details['state']
            result['model_geocoded_district'] = location_details['district']
        else:
            result['model_lat'] = None
            result['model_lon'] = None
            result['model_geocoded_state'] = None
            result['model_geocoded_district'] = None
    else:
        result['model_lat'] = None
        result['model_lon'] = None
        result['model_geocoded_state'] = None
        result['model_geocoded_district'] = None
    
    # Geocode ground truth (human baseline)
    parsed_gt = parse_structured_to_address_and_components(row['ground_truth'])
    
    if parsed_gt:
        location_details_gt = get_location_details(parsed_gt['address'], parsed_gt['components'])
        if location_details_gt:
            result['human_lat'] = location_details_gt['latitude']
            result['human_lon'] = location_details_gt['longitude']
            result['human_geocoded_state'] = location_details_gt['state']
            result['human_geocoded_district'] = location_details_gt['district']
        else:
            result['human_lat'] = None
            result['human_lon'] = None
            result['human_geocoded_state'] = None
            result['human_geocoded_district'] = None
    else:
        result['human_lat'] = None
        result['human_lon'] = None
        result['human_geocoded_state'] = None
        result['human_geocoded_district'] = None
    
    geocoding_results.append(result)
    
    # Progress indicator
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(test_predictions)} rows...")

geocoded_df = pd.DataFrame(geocoding_results)
print(f"\n✅ Geocoding complete!")
print(f"   Model predictions geocoded: {geocoded_df['model_lat'].notna().sum()}/{len(geocoded_df)}")
print(f"   Human baseline geocoded: {geocoded_df['human_lat'].notna().sum()}/{len(geocoded_df)}")

display(geocoded_df[['model_prediction', 'model_lat', 'model_lon', 'human_lat', 'human_lon']].head(10))

### 3.5 Calculate Distance Metrics

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points on Earth (in km).
    """
    if any(x is None for x in [lat1, lon1, lat2, lon2]):
        return None
    
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    # Earth's radius in kilometers
    R = 6371.0
    
    return R * c

# Calculate distances
geocoded_df['distance_km'] = geocoded_df.apply(
    lambda row: haversine_distance(
        row['model_lat'], row['model_lon'],
        row['human_lat'], row['human_lon']
    ),
    axis=1
)

# Filter to only rows where both model and human were successfully geocoded
valid_comparisons = geocoded_df[geocoded_df['distance_km'].notna()].copy()

print(f"Valid comparisons (both model and human geocoded): {len(valid_comparisons)}/{len(geocoded_df)}")
print(f"\nDistance Statistics:")
print(f"  Mean distance: {valid_comparisons['distance_km'].mean():.2f} km")
print(f"  Median distance: {valid_comparisons['distance_km'].median():.2f} km")
print(f"  Min distance: {valid_comparisons['distance_km'].min():.2f} km")
print(f"  Max distance: {valid_comparisons['distance_km'].max():.2f} km")
print(f"  Std deviation: {valid_comparisons['distance_km'].std():.2f} km")

# Distance buckets for practical interpretation
print(f"\nDistance Distribution:")
print(f"  Within 10 km:   {(valid_comparisons['distance_km'] <= 10).sum()} ({(valid_comparisons['distance_km'] <= 10).sum()/len(valid_comparisons)*100:.1f}%)")
print(f"  Within 50 km:   {(valid_comparisons['distance_km'] <= 50).sum()} ({(valid_comparisons['distance_km'] <= 50).sum()/len(valid_comparisons)*100:.1f}%)")
print(f"  Within 100 km:  {(valid_comparisons['distance_km'] <= 100).sum()} ({(valid_comparisons['distance_km'] <= 100).sum()/len(valid_comparisons)*100:.1f}%)")
print(f"  Beyond 100 km:  {(valid_comparisons['distance_km'] > 100).sum()} ({(valid_comparisons['distance_km'] > 100).sum()/len(valid_comparisons)*100:.1f}%)")

display(valid_comparisons[['model_prediction', 'ground_truth', 'distance_km']].head(10))

### 3.5.1 Analyze Geocoding Failures

In [ ]:
# Analyze why geocoding failed for some cases
print("="*80)
print("GEOCODING FAILURE ANALYSIS")
print("="*80)

# Breakdown of failures
model_failed = geocoded_df['model_lat'].isna().sum()
human_failed = geocoded_df['human_lat'].isna().sum()
both_succeeded = (geocoded_df['model_lat'].notna() & geocoded_df['human_lat'].notna()).sum()
both_failed = (geocoded_df['model_lat'].isna() & geocoded_df['human_lat'].isna()).sum()
only_model_failed = (geocoded_df['model_lat'].isna() & geocoded_df['human_lat'].notna()).sum()
only_human_failed = (geocoded_df['model_lat'].notna() & geocoded_df['human_lat'].isna()).sum()

print(f"\nTotal test cases: {len(geocoded_df)}")
print(f"\nGeocoding Success Rates:")
print(f"  Both succeeded:        {both_succeeded} ({both_succeeded/len(geocoded_df)*100:.1f}%)")
print(f"  Both failed:           {both_failed} ({both_failed/len(geocoded_df)*100:.1f}%)")
print(f"  Only model failed:     {only_model_failed} ({only_model_failed/len(geocoded_df)*100:.1f}%)")
print(f"  Only human failed:     {only_human_failed} ({only_human_failed/len(geocoded_df)*100:.1f}%)")

print(f"\nModel geocoding:  {len(geocoded_df) - model_failed}/{len(geocoded_df)} succeeded ({(len(geocoded_df) - model_failed)/len(geocoded_df)*100:.1f}%)")
print(f"Human geocoding:  {len(geocoded_df) - human_failed}/{len(geocoded_df)} succeeded ({(len(geocoded_df) - human_failed)/len(geocoded_df)*100:.1f}%)")

# Look at examples where both failed
print(f"\n" + "="*80)
print("EXAMPLES WHERE BOTH MODEL AND HUMAN FAILED TO GEOCODE")
print("="*80)
both_failed_df = geocoded_df[(geocoded_df['model_lat'].isna()) & (geocoded_df['human_lat'].isna())]
print(f"\nTotal cases where both failed: {len(both_failed_df)}")
if len(both_failed_df) > 0:
    print("\nSample cases:")
    display(both_failed_df[['model_prediction', 'ground_truth']].head(10))

# Look at examples where only model failed
print(f"\n" + "="*80)
print("EXAMPLES WHERE ONLY MODEL FAILED (HUMAN SUCCEEDED)")
print("="*80)
only_model_failed_df = geocoded_df[(geocoded_df['model_lat'].isna()) & (geocoded_df['human_lat'].notna())]
print(f"\nTotal cases: {len(only_model_failed_df)}")
if len(only_model_failed_df) > 0:
    print("\nSample cases (model extraction may be incomplete/wrong):")
    display(only_model_failed_df[['model_prediction', 'ground_truth']].head(10))

# Look at examples where only human failed
print(f"\n" + "="*80)
print("EXAMPLES WHERE ONLY HUMAN FAILED (MODEL SUCCEEDED)")
print("="*80)
only_human_failed_df = geocoded_df[(geocoded_df['model_lat'].notna()) & (geocoded_df['human_lat'].isna())]
print(f"\nTotal cases: {len(only_human_failed_df)}")
if len(only_human_failed_df) > 0:
    print("\nSample cases (human annotation may have issues):")
    display(only_human_failed_df[['model_prediction', 'ground_truth']].head(10))

### 3.6 Save Geocoding Results

In [ ]:
# Save the full geocoded results
try:
    saved_path = save_dataframe_csv(geocoded_df, "location_extraction_geocoding_results.csv", task_name=TASK_NAME)
    print(f"✅ Geocoding results saved to: {saved_path}")
except NameError:
    output_path = "./location_extraction_geocoding_results.csv"
    geocoded_df.to_csv(output_path, index=False)
    print(f"✅ Geocoding results saved to: {output_path}")

# Save summary metrics
summary_metrics = {
    'total_test_cases': len(geocoded_df),
    'model_geocoded_successfully': geocoded_df['model_lat'].notna().sum(),
    'human_geocoded_successfully': geocoded_df['human_lat'].notna().sum(),
    'valid_comparisons': len(valid_comparisons),
    'mean_distance_km': valid_comparisons['distance_km'].mean() if len(valid_comparisons) > 0 else None,
    'median_distance_km': valid_comparisons['distance_km'].median() if len(valid_comparisons) > 0 else None,
    'within_10km': (valid_comparisons['distance_km'] <= 10).sum() if len(valid_comparisons) > 0 else 0,
    'within_50km': (valid_comparisons['distance_km'] <= 50).sum() if len(valid_comparisons) > 0 else 0,
    'within_100km': (valid_comparisons['distance_km'] <= 100).sum() if len(valid_comparisons) > 0 else 0,
    'beyond_100km': (valid_comparisons['distance_km'] > 100).sum() if len(valid_comparisons) > 0 else 0,
}

summary_df = pd.DataFrame([summary_metrics])

try:
    saved_path = save_dataframe_csv(summary_df, "location_extraction_geocoding_metrics.csv", task_name=TASK_NAME)
    print(f"✅ Geocoding metrics saved to: {saved_path}")
except NameError:
    output_path = "./location_extraction_geocoding_metrics.csv"
    summary_df.to_csv(output_path, index=False)
    print(f"✅ Geocoding metrics saved to: {output_path}")

display(summary_df)

### 3.7 Visualize Results (Optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot distance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(valid_comparisons['distance_km'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Distances: Model vs Human Baseline')
axes[0].axvline(valid_comparisons['distance_km'].median(), color='red', linestyle='--', 
                label=f"Median: {valid_comparisons['distance_km'].median():.1f} km")
axes[0].legend()

# Cumulative distribution
sorted_distances = np.sort(valid_comparisons['distance_km'])
cumulative = np.arange(1, len(sorted_distances) + 1) / len(sorted_distances) * 100
axes[1].plot(sorted_distances, cumulative, linewidth=2)
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Cumulative % of Test Cases')
axes[1].set_title('Cumulative Distance Distribution')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(50, color='red', linestyle='--', alpha=0.5, label='50th percentile')
axes[1].axhline(90, color='orange', linestyle='--', alpha=0.5, label='90th percentile')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print percentiles
print("\nDistance Percentiles:")
for p in [10, 25, 50, 75, 90, 95]:
    val = np.percentile(valid_comparisons['distance_km'], p)
    print(f"  {p}th percentile: {val:.2f} km")

### 3.8 Compare Model Geocodes vs. Manual Baseline

This section compares the T5 model's geocoded predictions against the **original manual baseline coordinates** that were looked up by humans directly in Google Maps (not re-geocoded through the API).

#### 3.8.1 Load Baseline Data

In [ ]:
# Load the original dataset with manual baseline coordinates
# Now we can use incident_number to match since it's preserved throughout the workflow!
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info_augmented.csv"
baseline_data = pd.read_csv(url, dtype=str)

# Convert latitude/longitude to float
baseline_data['baseline_lat'] = pd.to_numeric(baseline_data['latitude'], errors='coerce')
baseline_data['baseline_lon'] = pd.to_numeric(baseline_data['longitude'], errors='coerce')

print(f"✅ Loaded baseline data: {len(baseline_data)} total incidents")
print(f"   Manual coordinates available: {baseline_data['baseline_lat'].notna().sum()} incidents")

# Merge geocoded results with baseline coordinates using incident_number
print("\n🔗 Matching by incident_number...")

# Ensure both incident_number columns are the same type (string) before merging
geocoded_df['incident_number'] = geocoded_df['incident_number'].astype(str)
baseline_data['incident_number'] = baseline_data['incident_number'].astype(str)

geocoded_with_baseline = geocoded_df.merge(
    baseline_data[['incident_number', 'baseline_lat', 'baseline_lon']],
    on='incident_number',
    how='left'
)

# Verify the merge
print(f"\n✅ Matched {geocoded_with_baseline['baseline_lat'].notna().sum()} test cases with manual baseline coordinates")
print(f"   Total test cases: {len(geocoded_with_baseline)}")
print(f"   Match rate: {geocoded_with_baseline['baseline_lat'].notna().sum() / len(geocoded_with_baseline) * 100:.1f}%")

# Display sample to verify correct matching
print("\n📊 Sample of matched data:")
display(geocoded_with_baseline[['incident_number', 'incident_summary', 'baseline_lat', 'baseline_lon', 'model_lat', 'model_lon']].head())

#### 3.8.2 Calculate distance between model geocodes and manual baseline

In [ ]:
# Calculate distance between model geocodes and manual baseline
geocoded_with_baseline['distance_to_baseline_km'] = geocoded_with_baseline.apply(
    lambda row: haversine_distance(
        row['model_lat'], row['model_lon'],
        row['baseline_lat'], row['baseline_lon']
    ),
    axis=1
)

# Filter to cases where both model and baseline have coordinates
valid_baseline_comparisons = geocoded_with_baseline[
    geocoded_with_baseline['distance_to_baseline_km'].notna()
].copy()

print("="*80)
print("MODEL GEOCODES VS. MANUAL BASELINE COMPARISON")
print("="*80)
print(f"\nValid comparisons (both model and baseline available): {len(valid_baseline_comparisons)}/{len(geocoded_with_baseline)}")

if len(valid_baseline_comparisons) > 0:
    print(f"\nDistance Statistics (Model vs. Manual Baseline):")
    print(f"  Mean distance:   {valid_baseline_comparisons['distance_to_baseline_km'].mean():.2f} km")
    print(f"  Median distance: {valid_baseline_comparisons['distance_to_baseline_km'].median():.2f} km")
    print(f"  Min distance:    {valid_baseline_comparisons['distance_to_baseline_km'].min():.2f} km")
    print(f"  Max distance:    {valid_baseline_comparisons['distance_to_baseline_km'].max():.2f} km")
    print(f"  Std deviation:   {valid_baseline_comparisons['distance_to_baseline_km'].std():.2f} km")
    
    print(f"\nDistance Distribution (Practical Accuracy):")
    print(f"  Within 10 km:   {(valid_baseline_comparisons['distance_to_baseline_km'] <= 10).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] <= 10).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    print(f"  Within 50 km:   {(valid_baseline_comparisons['distance_to_baseline_km'] <= 50).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] <= 50).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    print(f"  Within 100 km:  {(valid_baseline_comparisons['distance_to_baseline_km'] <= 100).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] <= 100).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    print(f"  Beyond 100 km:  {(valid_baseline_comparisons['distance_to_baseline_km'] > 100).sum()} ({(valid_baseline_comparisons['distance_to_baseline_km'] > 100).sum()/len(valid_baseline_comparisons)*100:.1f}%)")
    
    print("\nPercentiles:")
    for p in [10, 25, 50, 75, 90, 95]:
        val = np.percentile(valid_baseline_comparisons['distance_to_baseline_km'], p)
        print(f"  {p}th percentile: {val:.2f} km")
    
    print("\nSample comparisons:")
    display(valid_baseline_comparisons[['model_prediction', 'model_lat', 'model_lon', 'baseline_lat', 'baseline_lon', 'distance_to_baseline_km']].head(10))
else:
    print("\n⚠️  No valid comparisons available. Check data matching.")

#### 3.8.3 Save Results with Baseline Comparison

In [ ]:
# Save the results with baseline comparison
try:
    saved_path = save_dataframe_csv(geocoded_with_baseline, "location_extraction_geocoding_with_baseline.csv", task_name=TASK_NAME)
    print(f"✅ Results with baseline comparison saved to: {saved_path}")
except NameError:
    output_path = "./location_extraction_geocoding_with_baseline.csv"
    geocoded_with_baseline.to_csv(output_path, index=False)
    print(f"✅ Results with baseline comparison saved to: {output_path}")

# Save summary metrics comparing to baseline
if len(valid_baseline_comparisons) > 0:
    baseline_summary = {
        'total_test_cases': len(geocoded_with_baseline),
        'valid_comparisons_to_baseline': len(valid_baseline_comparisons),
        'mean_distance_to_baseline_km': valid_baseline_comparisons['distance_to_baseline_km'].mean(),
        'median_distance_to_baseline_km': valid_baseline_comparisons['distance_to_baseline_km'].median(),
        'within_10km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] <= 10).sum(),
        'within_50km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] <= 50).sum(),
        'within_100km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] <= 100).sum(),
        'beyond_100km_baseline': (valid_baseline_comparisons['distance_to_baseline_km'] > 100).sum(),
    }
    
    baseline_summary_df = pd.DataFrame([baseline_summary])
    
    try:
        saved_path = save_dataframe_csv(baseline_summary_df, "location_extraction_baseline_metrics.csv", task_name=TASK_NAME)
        print(f"✅ Baseline comparison metrics saved to: {saved_path}")
    except NameError:
        output_path = "./location_extraction_baseline_metrics.csv"
        baseline_summary_df.to_csv(output_path, index=False)
        print(f"✅ Baseline comparison metrics saved to: {output_path}")
    
    display(baseline_summary_df)

### 3.9 Accuracy by Geographic Specificity Level

Analyze how geocoding accuracy varies by the level of geographic detail extracted (village-level vs. district-level vs. state-level).

#### 3.9.1 Classify Geographic Level

In [ ]:
# Classify each prediction by geographic specificity level
def classify_geographic_level(prediction_text):
    """
    Classify the geographic specificity of a prediction.
    Returns: 'village', 'district', 'state', or 'none'
    """
    if not prediction_text or prediction_text.strip() == '':
        return 'none'
    
    parsed = parse_structured_location(prediction_text)
    
    # Most specific: Has village or other_locations
    if parsed['village'] or parsed['other_locations']:
        return 'village'
    # Medium: Has district but no village
    elif parsed['district']:
        return 'district'
    # Least specific: Has only state
    elif parsed['state']:
        return 'state'
    else:
        return 'none'

# Add geographic level classification to our results
if len(valid_baseline_comparisons) > 0:
    valid_baseline_comparisons['geographic_level'] = valid_baseline_comparisons['model_prediction'].apply(classify_geographic_level)
    
    print("="*80)
    print("ACCURACY BY GEOGRAPHIC SPECIFICITY LEVEL")
    print("="*80)
    
    # Count by level
    level_counts = valid_baseline_comparisons['geographic_level'].value_counts()
    print(f"\nDistribution of predictions by geographic level:")
    for level in ['village', 'district', 'state', 'none']:
        count = level_counts.get(level, 0)
        pct = (count / len(valid_baseline_comparisons) * 100) if len(valid_baseline_comparisons) > 0 else 0
        print(f"  {level.capitalize():12} level: {count:4d} ({pct:5.1f}%)")
    
    # Analyze accuracy by level
    print(f"\n{'='*80}")
    print("DISTANCE METRICS BY GEOGRAPHIC LEVEL")
    print(f"{'='*80}\n")
    
    for level in ['village', 'district', 'state']:
        level_data = valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]
        
        if len(level_data) > 0:
            print(f"\n{level.upper()}-LEVEL PREDICTIONS (n={len(level_data)}):")
            print(f"  Mean distance:   {level_data['distance_to_baseline_km'].mean():.2f} km")
            print(f"  Median distance: {level_data['distance_to_baseline_km'].median():.2f} km")
            print(f"  Std deviation:   {level_data['distance_to_baseline_km'].std():.2f} km")
            print(f"  Within 10 km:    {(level_data['distance_to_baseline_km'] <= 10).sum()} ({(level_data['distance_to_baseline_km'] <= 10).sum()/len(level_data)*100:.1f}%)")
            print(f"  Within 50 km:    {(level_data['distance_to_baseline_km'] <= 50).sum()} ({(level_data['distance_to_baseline_km'] <= 50).sum()/len(level_data)*100:.1f}%)")
            print(f"  Within 100 km:   {(level_data['distance_to_baseline_km'] <= 100).sum()} ({(level_data['distance_to_baseline_km'] <= 100).sum()/len(level_data)*100:.1f}%)")
        else:
            print(f"\n{level.upper()}-LEVEL: No predictions at this level")
    
else:
    print("⚠️  No valid baseline comparisons available for stratification analysis")

#### 3.9.2 Visualize Accuracy by Geographic Level

In [ ]:
# Visualize accuracy by geographic level
if len(valid_baseline_comparisons) > 0 and 'geographic_level' in valid_baseline_comparisons.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Box plot of distances by level
    level_order = ['village', 'district', 'state']
    data_by_level = [
        valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]['distance_to_baseline_km'].dropna()
        for level in level_order
    ]
    
    bp = axes[0].boxplot(data_by_level, labels=[l.capitalize() for l in level_order], 
                         patch_artist=True, showmeans=True)
    axes[0].set_ylabel('Distance to Baseline (km)')
    axes[0].set_xlabel('Geographic Specificity Level')
    axes[0].set_title('Distance Distribution by Geographic Level')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Color the boxes
    colors = ['lightgreen', 'lightblue', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    # Bar chart of accuracy rates (% within thresholds)
    thresholds = [10, 50, 100]
    x = np.arange(len(level_order))
    width = 0.25
    
    for i, threshold in enumerate(thresholds):
        rates = []
        for level in level_order:
            level_data = valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]
            if len(level_data) > 0:
                rate = (level_data['distance_to_baseline_km'] <= threshold).sum() / len(level_data) * 100
                rates.append(rate)
            else:
                rates.append(0)
        
        axes[1].bar(x + i*width, rates, width, label=f'Within {threshold} km')
    
    axes[1].set_xlabel('Geographic Specificity Level')
    axes[1].set_ylabel('% of Predictions')
    axes[1].set_title('Accuracy Rate by Geographic Level')
    axes[1].set_xticks(x + width)
    axes[1].set_xticklabels([l.capitalize() for l in level_order])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*80)
    print("KEY INSIGHT:")
    print("="*80)
    print("Village-level predictions should be more accurate (lower median distance)")
    print("District-level predictions will have higher error (using district centroid)")
    print("State-level predictions will have highest error (using state centroid)")
else:
    print("⚠️  Cannot create visualizations - no data with geographic levels")

#### 3.9.3 Save Accuracy by Geographic Level

In [ ]:
# Save stratified results
if len(valid_baseline_comparisons) > 0 and 'geographic_level' in valid_baseline_comparisons.columns:
    # Create summary by level
    level_summary = []
    for level in ['village', 'district', 'state']:
        level_data = valid_baseline_comparisons[valid_baseline_comparisons['geographic_level'] == level]
        if len(level_data) > 0:
            level_summary.append({
                'geographic_level': level,
                'count': len(level_data),
                'mean_distance_km': level_data['distance_to_baseline_km'].mean(),
                'median_distance_km': level_data['distance_to_baseline_km'].median(),
                'std_distance_km': level_data['distance_to_baseline_km'].std(),
                'pct_within_10km': (level_data['distance_to_baseline_km'] <= 10).sum() / len(level_data) * 100,
                'pct_within_50km': (level_data['distance_to_baseline_km'] <= 50).sum() / len(level_data) * 100,
                'pct_within_100km': (level_data['distance_to_baseline_km'] <= 100).sum() / len(level_data) * 100,
            })
    
    level_summary_df = pd.DataFrame(level_summary)
    
    try:
        saved_path = save_dataframe_csv(level_summary_df, "location_extraction_accuracy_by_level.csv", task_name=TASK_NAME)
        print(f"✅ Accuracy by geographic level saved to: {saved_path}")
    except NameError:
        output_path = "./location_extraction_accuracy_by_level.csv"
        level_summary_df.to_csv(output_path, index=False)
        print(f"✅ Accuracy by geographic level saved to: {output_path}")
    
    display(level_summary_df)